# Iodine Clock Reaction — Data Collection

Record **HSV color vs. time** curves during iodine clock reactions using a USB camera.

**Workflow:**
1. Enter an experiment name (used as the filename for saved files)
2. Adjust **Camera ID** and **ROI** (Region of Interest) to frame the reaction vessel
3. Click **Start Recording** — the camera opens and data collection begins
4. Click **Stop Recording** — recording stops and the camera closes
5. Click **Save** — writes a `.avi` video and a `.xlsx` Excel file containing HSV data

> **HSV scale used (standard, NOT OpenCV):** $$H \in [0, 360°]$$, $$S \in [0, 100\%]$$, $$V \in [0, 100\%]$$

In [ ]:
import cv2
import numpy as np
import threading
import time
import os

import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [6]:
class IodineClockRecorder:
    """Records iodine clock reaction color changes via a USB camera."""

    def __init__(self, color_cam_id=1, target_fps=15):
        self.color_cam_id = color_cam_id
        self.target_fps = target_fps

        self._cap = None
        self._running = False
        self._recording = False
        self._thread = None

        # ROI coordinates (pixels)
        self.roi_x1 = 200
        self.roi_y1 = 200
        self.roi_x2 = 400
        self.roi_y2 = 400

        # Recorded data
        self.hsv_data = []
        self._start_time = None
        self._video_writer = None
        self._video_path = None

        # UI widget references (assigned externally)
        self.image_widget = None
        self.status_label = None

    # ------------------------------------------------------------------ helpers
    def _update_status(self, msg):
        if self.status_label is not None:
            self.status_label.value = f"<b>Status:</b> {msg}"

    def _avg_hsv_in_roi(self, frame):
        """Return average (H, S, V) inside the ROI in standard scale."""
        fh, fw = frame.shape[:2]
        x1, x2 = np.clip(self.roi_x1, 0, fw - 1), np.clip(self.roi_x2, 1, fw)
        y1, y2 = np.clip(self.roi_y1, 0, fh - 1), np.clip(self.roi_y2, 1, fh)

        roi = frame[y1:y2, x1:x2]
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

        h_cv = float(np.mean(hsv[:, :, 0]))
        s_cv = float(np.mean(hsv[:, :, 1]))
        v_cv = float(np.mean(hsv[:, :, 2]))

        # OpenCV HSV (H:0-179, S:0-255, V:0-255) → standard (H:0-360, S:0-100, V:0-100)
        return round(h_cv * 2.0, 2), round(s_cv / 2.55, 2), round(v_cv / 2.55, 2)

    # ----------------------------------------------------------- camera thread
    def _loop(self):
        interval = 1.0 / self.target_fps

        while self._running:
            t0 = time.time()
            ret, frame = self._cap.read()
            if not ret or frame is None:
                self._update_status("Camera read error!")
                break

            h, s, v = self._avg_hsv_in_roi(frame)

            # Display frame (with overlay)
            disp = frame.copy()
            cv2.rectangle(disp,
                          (self.roi_x1, self.roi_y1),
                          (self.roi_x2, self.roi_y2), (0, 255, 0), 2)
            cv2.putText(disp, f"H:{h:.1f}  S:{s:.1f}  V:{v:.1f}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

            if self._recording:
                elapsed = time.time() - self._start_time
                self.hsv_data.append(
                    {"time_s": round(elapsed, 3), "H": h, "S": s, "V": v}
                )
                if self._video_writer is not None:
                    self._video_writer.write(frame)   # raw frame (no overlay)

                # REC indicator on the display copy only
                cv2.circle(disp, (disp.shape[1] - 30, 30), 10, (0, 0, 255), -1)
                cv2.putText(disp, f"REC  {elapsed:.1f}s",
                            (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            ok, jpg = cv2.imencode(".jpg", disp, [cv2.IMWRITE_JPEG_QUALITY, 85])
            if ok and self.image_widget is not None:
                self.image_widget.value = jpg.tobytes()

            dt = time.time() - t0
            if dt < interval:
                time.sleep(interval - dt)

        # cleanup on thread exit
        if self._video_writer is not None:
            self._video_writer.release()
            self._video_writer = None
        if self._cap is not None:
            self._cap.release()
            self._cap = None

    # -------------------------------------------------------------- public API
    def start_recording(self, exp_name, save_dir="."):
        if self._recording:
            self._update_status("Already recording!")
            return
        if not exp_name or not exp_name.strip():
            self._update_status("Please enter an experiment name first!")
            return

        self.hsv_data = []

        self._cap = cv2.VideoCapture(self.color_cam_id)
        if not self._cap.isOpened():
            self._update_status(f"Cannot open camera (ID={self.color_cam_id})!")
            return

        ret, test_frame = self._cap.read()
        if not ret or test_frame is None:
            self._update_status("Cannot read from camera!")
            self._cap.release()
            self._cap = None
            return

        fh, fw = test_frame.shape[:2]
        os.makedirs(save_dir, exist_ok=True)
        self._video_path = os.path.join(save_dir, f"{exp_name.strip()}.avi")
        fourcc = cv2.VideoWriter_fourcc(*"XVID")
        self._video_writer = cv2.VideoWriter(
            self._video_path, fourcc, self.target_fps, (fw, fh)
        )

        self._recording = True
        self._running = True
        self._start_time = time.time()

        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        self._update_status(f"Recording '{exp_name.strip()}' …")

    def stop_recording(self):
        if not self._recording:
            self._update_status("Not recording.")
            return

        self._recording = False
        self._running = False
        if self._thread is not None:
            self._thread.join(timeout=5)
            self._thread = None

        n = len(self.hsv_data)
        dur = self.hsv_data[-1]["time_s"] if self.hsv_data else 0
        self._update_status(f"Stopped. {n} data points over {dur:.1f} s.  "
                            f"Video saved to {self._video_path}")

    def save_data(self, exp_name, save_dir="."):
        if self._recording:
            self._update_status("Stop recording before saving!")
            return False
        if not exp_name or not exp_name.strip():
            self._update_status("Please enter an experiment name!")
            return False
        if not self.hsv_data:
            self._update_status("No recorded data to save!")
            return False

        os.makedirs(save_dir, exist_ok=True)
        xlsx_path = os.path.join(save_dir, f"{exp_name.strip()}.xlsx")

        df = pd.DataFrame(self.hsv_data)
        df.columns = ["Time (s)", "H (0-360)", "S (0-100)", "V (0-100)"]
        df.to_excel(xlsx_path, index=False, sheet_name="HSV Data")

        self._update_status(
            f"Saved!  Video → {self._video_path}  |  Excel → {xlsx_path}"
        )
        return True

In [7]:
recorder = IodineClockRecorder(color_cam_id=1, target_fps=15)

# ── widgets ──────────────────────────────────────────────────────────────────
w_lay = widgets.Layout(width="300px")
short_lay = widgets.Layout(width="180px")
desc_style = {"description_width": "80px"}
roi_style = {"description_width": "60px"}

exp_name_w = widgets.Text(
    value="", placeholder="e.g. iodine_0.1M_trial1",
    description="Exp Name:", layout=w_lay, style=desc_style,
)
cam_id_w = widgets.IntText(
    value=1, description="Cam ID:", layout=short_lay, style=desc_style,
)

roi_x1_w = widgets.IntText(value=200, description="ROI x1:", layout=short_lay, style=roi_style)
roi_y1_w = widgets.IntText(value=200, description="ROI y1:", layout=short_lay, style=roi_style)
roi_x2_w = widgets.IntText(value=400, description="ROI x2:", layout=short_lay, style=roi_style)
roi_y2_w = widgets.IntText(value=400, description="ROI y2:", layout=short_lay, style=roi_style)

btn_lay = widgets.Layout(width="180px", height="36px")
start_btn = widgets.Button(description="▶ Start Recording", button_style="success", layout=btn_lay)
stop_btn  = widgets.Button(description="■ Stop Recording",  button_style="danger",  layout=btn_lay)
save_btn  = widgets.Button(description="💾 Save",            button_style="info",    layout=btn_lay)

recorder.image_widget  = widgets.Image(format="jpeg", width=640, height=480)
recorder.status_label  = widgets.HTML("<b>Status:</b> Ready")
output_area = widgets.Output()

# ── callbacks ────────────────────────────────────────────────────────────────
def _sync_params():
    recorder.color_cam_id = cam_id_w.value
    recorder.roi_x1 = roi_x1_w.value
    recorder.roi_y1 = roi_y1_w.value
    recorder.roi_x2 = roi_x2_w.value
    recorder.roi_y2 = roi_y2_w.value

def on_start(b):
    _sync_params()
    recorder.start_recording(exp_name_w.value)

def on_stop(b):
    recorder.stop_recording()

def on_save(b):
    ok = recorder.save_data(exp_name_w.value)
    if ok:
        with output_area:
            output_area.clear_output(wait=True)
            df = pd.DataFrame(recorder.hsv_data)
            fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
            keys   = ["H", "S", "V"]
            labels = ["H (0‑360°)", "S (0‑100%)", "V (0‑100%)"]
            colors = ["tab:red", "tab:green", "tab:blue"]
            for ax, key, label, c in zip(axes, keys, labels, colors):
                ax.plot(df["time_s"], df[key], color=c, linewidth=1)
                ax.set_ylabel(label)
                ax.grid(True, alpha=0.3)
            axes[-1].set_xlabel("Time (s)")
            fig.suptitle(f"HSV vs Time — {exp_name_w.value}", fontsize=14)
            plt.tight_layout()
            plt.show()

start_btn.on_click(on_start)
stop_btn.on_click(on_stop)
save_btn.on_click(on_save)

# ── layout ───────────────────────────────────────────────────────────────────
settings_row = widgets.HBox([
    widgets.VBox([exp_name_w, cam_id_w]),
    widgets.VBox([
        widgets.HTML("<b>ROI (Region of Interest)</b>"),
        widgets.HBox([roi_x1_w, roi_y1_w]),
        widgets.HBox([roi_x2_w, roi_y2_w]),
    ]),
])

buttons_row = widgets.HBox([start_btn, stop_btn, save_btn])

save_note = widgets.HTML(
    "<i style='color:#666; font-size:12px;'>"
    "Files (.avi and .xlsx) are saved to the same directory as this notebook.</i>"
)

ui = widgets.VBox([
    settings_row,
    widgets.HTML("<hr style='border:1px solid #ccc'>"),
    buttons_row,
    save_note,
    recorder.status_label,
    recorder.image_widget,
    output_area,
])

display(ui)

PermissionError: [Errno 13] Permission denied: '.\\1.xlsx'